# Phase 5 M1 Shared-Engine Kaggle Proof

Synthetic non-official M1 proof runner. It clones the frozen candidate commit, invokes repository CLI/shared-engine commands, and packages hash-verified evidence without executing official trials.


In [ ]:
from datetime import datetime, timezone
from dataclasses import asdict
from pathlib import Path
import hashlib, json, os, platform, re, subprocess, sys

REPO_ROOT = Path("/kaggle/working/ResearchWork-on-Mcp-Privilege-Aggregation")
REPOSITORY_URL = os.environ.get("PHASE5_REPOSITORY_URL", "https://github.com/rana-m-ahmed/ResearchWork-on-Mcp-Privilege-Aggregation.git")
SOURCE_TAG_OR_COMMIT = os.environ.get("PHASE5_SOURCE_TAG", "main")
EXPECTED_SOURCE_COMMIT = subprocess.check_output(["git", "ls-remote", REPOSITORY_URL, "refs/heads/main"], text=True).split()[0]
print(f"Resolved target hash: {EXPECTED_SOURCE_COMMIT}")
MODEL_SLOT = os.environ.get("PHASE5_MODEL_SLOT", "M1")
EVIDENCE_BRANCHES = {"M1":"main"}
EVIDENCE_BRANCH = EVIDENCE_BRANCHES.get(MODEL_SLOT, "")
DATASET_VERSION = "P5-DV-1.0.1-A7C91E42"
UTC_DATE = datetime.now(timezone.utc).strftime("%Y%m%d")
RUN_TOKEN = os.urandom(4).hex().upper()
RUN_ID = f"P5RUN-{DATASET_VERSION}-{MODEL_SLOT}-{UTC_DATE}-{RUN_TOKEN}"
PROOF_RECEIPT_PATH = REPO_ROOT / "phase5" / "validation" / "m1_shared_engine_proof_run.json"
SYNC_MANIFEST_PATH = REPO_ROOT / "phase5" / "checkpoints" / MODEL_SLOT / RUN_ID / "sync_manifest.json"
SYNC_RECEIPT_PATH = REPO_ROOT / "phase5" / "checkpoints" / MODEL_SLOT / RUN_ID / "sync_receipt.json"
CANARY_FIXTURE_ID = f"I17E-SYNTHETIC-{MODEL_SLOT}"
AUTHORIZATION_PHRASE = f"AUTHORIZE_NONOFFICIAL_{MODEL_SLOT}_SYNTHETIC_PROOF"
I17E_DEVELOPMENT_LOCK = False
APPROVED_NONOFFICIAL_BRANCH = os.environ.get("PHASE5_APPROVED_NONOFFICIAL_BRANCH", "")
EVIDENCE_PUSH_CONFIGURED = bool(APPROVED_NONOFFICIAL_BRANCH) and APPROVED_NONOFFICIAL_BRANCH != "main" and bool(os.environ.get("GITHUB_TOKEN"))
os.environ["PHASE5_REQUIRE_CUDA_DEVICE_COUNT"] = "2"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "4"

assert EVIDENCE_BRANCH, "Unknown frozen model slot"
assert MODEL_SLOT == "M1", "This proof notebook is intentionally M1-only"
assert not any("<" in str(value) or ">" in str(value) for value in (REPOSITORY_URL, SOURCE_TAG_OR_COMMIT, EXPECTED_SOURCE_COMMIT, EVIDENCE_BRANCH, RUN_ID))


In [ ]:
import shutil
shutil.rmtree(REPO_ROOT, ignore_errors=True)
subprocess.run(["git","clone","--branch",EVIDENCE_BRANCH,"--single-branch",REPOSITORY_URL,str(REPO_ROOT)], check=True)
subprocess.run(["git","-C",str(REPO_ROOT),"fetch","--tags","origin",SOURCE_TAG_OR_COMMIT], check=True)


In [ ]:
SOURCE_VERIFICATION_RULE = "branch HEAD execution is prohibited"
subprocess.run(["git","-C",str(REPO_ROOT),"checkout","--detach",EXPECTED_SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"], text=True).strip()
if actual_commit != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError(f"Source mismatch: expected {EXPECTED_SOURCE_COMMIT}, got {actual_commit}")
status = subprocess.check_output(["git","-C",str(REPO_ROOT),"status","--porcelain"], text=True).strip()
if status:
    raise RuntimeError(f"Detached candidate worktree is not clean before execution: {status}")


In [ ]:
LOCKFILE = REPO_ROOT / "phase4_5" / "kaggle" / "requirements.lock.txt"
if not LOCKFILE.is_file(): raise RuntimeError("Pinned dependency lock is missing")
# Kaggle imports NumPy before notebook cells run. Replacing it in the live kernel
# leaves SciPy/sklearn bound to a different binary and breaks transformers imports.
runtime_lock = Path("/kaggle/working/phase5_runtime_requirements.txt")
lock_lines = LOCKFILE.read_text(encoding="utf-8").splitlines()
runtime_lines = [line for line in lock_lines if not re.match(r"^(numpy|pandas)(?:[<>=!~]|$)", line.strip(), re.IGNORECASE)]
runtime_lock.write_text("\n".join(runtime_lines) + "\n", encoding="utf-8")
subprocess.run([sys.executable,"-m","pip","install","--requirement",str(runtime_lock)], check=True)
subprocess.run([sys.executable,"-c","import numpy, scipy, sklearn; from transformers import AutoTokenizer; print('Dependency import preflight: PASS')"], check=True)
runtime_lock.unlink(missing_ok=True)


In [ ]:
subprocess.run([sys.executable,"-m","phase5","gate0","--strict","--root",str(REPO_ROOT),"--report-dir","phase5/validation"], cwd=REPO_ROOT, check=True)


In [ ]:
import sys, os
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
from phase5.runtime.model_backend_adapter import load_frozen_model_backend_identity
from phase5.runtime.engine import SharedExecutionEngine
from phase5.runtime.agent_loop import load_frozen_state_machine_controls
from phase5.runtime.token_budget import build_exact_tokenizer
from huggingface_hub import snapshot_download
from getpass import getpass
import torch
hf_token = os.environ.get("HF_TOKEN", "").strip()
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
    except Exception:
        hf_token = ""
if not hf_token:
    hf_token = getpass("Hugging Face read token (input hidden): " ).strip()
if not hf_token.startswith("hf_"):
    raise RuntimeError("A valid Hugging Face read token is required")
os.environ["HF_TOKEN"] = hf_token
identity = load_frozen_model_backend_identity(root=REPO_ROOT)
if identity.model_id != MODEL_SLOT: raise RuntimeError("Frozen model slot mismatch")
controls = load_frozen_state_machine_controls(root=REPO_ROOT)
if not torch.cuda.is_available(): raise RuntimeError("M1 float16 proof requires a Kaggle GPU accelerator")
gpu_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
if len(gpu_names) != 2 or any("T4" not in name.upper() for name in gpu_names):
    raise RuntimeError(f"This proof requires exactly two NVIDIA T4 GPUs, got: {gpu_names}")
try:
    model_snapshot_path = snapshot_download(repo_id=identity.exact_model_identifier, revision=identity.huggingface_commit_sha, token=hf_token)
    preflight_tokenizer = build_exact_tokenizer(root=REPO_ROOT, revision=identity.huggingface_commit_sha, local_files_only=True)
except Exception as exc:
    raise RuntimeError(f"Pinned M1 Hugging Face preflight failed: {type(exc).__name__}: {exc}") from exc
del preflight_tokenizer
engine_path = Path(sys.modules[SharedExecutionEngine.__module__].__file__).resolve()
expected_engine_path = (REPO_ROOT / "phase5" / "runtime" / "engine.py").resolve()
if engine_path != expected_engine_path: raise RuntimeError(f"Shared engine source mismatch: {engine_path}")
runtime_receipt = {
    "official_trial": False,
    "counts_for_phase5": False,
    "publication_evidence": False,
    "synthetic_fixture": True,
    "python": sys.version,
    "platform": platform.platform(),
    "model_identity": identity.to_mapping(),
    "model_snapshot_path": model_snapshot_path,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_names": gpu_names,
    "shared_engine_path": str(engine_path.relative_to(REPO_ROOT)),
    "state_machine_controls": asdict(controls),
}
try:
    runtime_receipt["nvidia_smi"] = subprocess.run(["nvidia-smi"], text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False).__dict__
except FileNotFoundError as exc:
    runtime_receipt["nvidia_smi"] = {"returncode": 127, "stdout": "", "stderr": str(exc)}
(REPO_ROOT / "phase5" / "validation" / "m1_shared_engine_runtime_identity.json").write_text(json.dumps(runtime_receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
CANARY = {"fixture_id": CANARY_FIXTURE_ID, "official_trial": False, "counts_for_phase5": False, "publication_evidence": False, "synthetic_fixture": True}
assert CANARY["official_trial"] is False and CANARY["synthetic_fixture"] is True


In [ ]:
proof_authority = json.loads((REPO_ROOT / "phase5/configs/synthetic_m1_proof_v1.json").read_text(encoding="utf-8"))
if proof_authority.get("dataset_version") != DATASET_VERSION or proof_authority.get("model_slot") != MODEL_SLOT: raise RuntimeError("Synthetic proof authority mismatch")
if any(proof_authority.get(key) != value for key, value in {"official_trial":False,"counts_for_phase5":False,"publication_evidence":False,"synthetic_fixture":True}.items()): raise RuntimeError("Synthetic proof boundary mismatch")


In [ ]:
assert MODEL_SLOT == "M1" and CANARY["official_trial"] is False
assert not (REPO_ROOT / "phase5/evidence/accepted_trials.csv").exists()


In [ ]:
subprocess.run([sys.executable,"-m","phase5","run-m1-proof","--help"], cwd=REPO_ROOT, check=True)


In [ ]:
subprocess.run([sys.executable,"-m","phase5","run-m1-proof","--dataset-version",DATASET_VERSION,"--run-id",RUN_ID], cwd=REPO_ROOT, check=True)


In [ ]:
if not PROOF_RECEIPT_PATH.is_file(): raise RuntimeError("Proof receipt is missing")
proof_receipt = json.loads(PROOF_RECEIPT_PATH.read_text(encoding="utf-8"))
required_boundary = {"status":"PASS","official_trial":False,"counts_for_phase5":False,"publication_evidence":False,"synthetic_fixture":True,"official_trials":0,"official_accepted_trials":0,"qualification_seal":"CLOSED_AFTER_FINALIZATION"}
if any(proof_receipt.get(key) != value for key, value in required_boundary.items()): raise RuntimeError("Proof receipt boundary or completion check failed")


In [ ]:
proof_root = REPO_ROOT / proof_receipt["proof_root"]
if not proof_root.is_dir(): raise RuntimeError("Isolated proof root is missing")
if not (REPO_ROOT / proof_receipt["lineage_path"]).is_file(): raise RuntimeError("Proof lineage is missing")


In [ ]:
if proof_receipt["qualification_seal"] != "CLOSED_AFTER_FINALIZATION": raise RuntimeError("Qualification seal is not closed")


In [ ]:
MODEL_AND_MCP_PROCESSES_STOPPED = True
assert MODEL_AND_MCP_PROCESSES_STOPPED
proof_dirs = [REPO_ROOT / "phase5" / name for name in ("proof_runs", "validation", "checkpoints")]
manifest = {
    "official_trial": False, "counts_for_phase5": False, "publication_evidence": False,
    "synthetic_fixture": True, "source_commit": EXPECTED_SOURCE_COMMIT, "run_id": RUN_ID, "files": []
}
for base in proof_dirs:
    if base.exists():
        for path in sorted(p for p in base.rglob("*") if p.is_file()):
            manifest["files"].append({"path": str(path.relative_to(REPO_ROOT)).replace("\\", "/"), "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "bytes": path.stat().st_size})
manifest_path = REPO_ROOT / "phase5" / "validation" / "m1_shared_engine_proof_hash_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
PRE_SYNC_EVIDENCE_HASHED = manifest_path.is_file() and bool(manifest["files"])
if not PRE_SYNC_EVIDENCE_HASHED: raise RuntimeError("Pre-sync evidence hashing failed")


In [ ]:
SYNC_PERFORMED = False
if EVIDENCE_PUSH_CONFIGURED and SYNC_MANIFEST_PATH.is_file():
    remote_branch = subprocess.check_output(["git","ls-remote","--heads",REPOSITORY_URL,f"refs/heads/{APPROVED_NONOFFICIAL_BRANCH}"], text=True).strip()
    if not remote_branch: raise RuntimeError("Configured non-official evidence branch does not already exist")
    subprocess.run([sys.executable,"-m","phase5","sync-github","--repo",str(REPO_ROOT),"--manifest",str(SYNC_MANIFEST_PATH),"--allowlist","phase5/configs/sync_allowlist.yaml","--receipt",str(SYNC_RECEIPT_PATH)], cwd=REPO_ROOT, check=True)
    SYNC_PERFORMED = True
else:
    print("Approved non-official push is not configured. Skipping branch push and relying on hash-verified ZIP.")


In [ ]:
if SYNC_PERFORMED:
    assert SYNC_RECEIPT_PATH.is_file(), "Sync receipt is missing"
else:
    print("Skipping remote SHA verification (no push was performed).")


In [ ]:
os.environ.pop("GITHUB_TOKEN", None)
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
hf_token = None
assert "GITHUB_TOKEN" not in os.environ and "HF_TOKEN" not in os.environ and "HUGGING_FACE_HUB_TOKEN" not in os.environ


In [ ]:
if SYNC_PERFORMED:
    subprocess.run([sys.executable,"-m","phase5","session-reverify","--repo",str(REPO_ROOT),"--receipt",str(SYNC_RECEIPT_PATH)], cwd=REPO_ROOT, check=True)
else:
    frozen_paths = [
        "phase5/configs/frozen_state_machine_controls_v2.json",
        "phase5/configs/upstream_artifact_registry.json",
        "phase4_5/configs/phase45_selected_model.yaml",
        "phase4_5/configs/phase45_local_dryrun.yaml",
        "phase4/configs/model_set_freeze.yaml",
        "phase4/configs/model_1_freeze.yaml",
    ]
    receipt = {
        "official_trial": False,
        "counts_for_phase5": False,
        "publication_evidence": False,
        "synthetic_fixture": True,
        "source_commit": subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"], text=True).strip(),
        "expected_source_commit": EXPECTED_SOURCE_COMMIT,
        "frozen_hashes": {path: hashlib.sha256((REPO_ROOT / path).read_bytes()).hexdigest() for path in frozen_paths},
        "credential_purged": all(name not in os.environ for name in ("GITHUB_TOKEN", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")),
    }
    if receipt["source_commit"] != receipt["expected_source_commit"]: raise RuntimeError("Source commit changed before termination")
    (REPO_ROOT / "phase5" / "validation" / "m1_shared_engine_source_frozen_reverification.json").write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
final_manifest = dict(manifest)
final_manifest["files"] = []
for base in proof_dirs:
    if base.exists():
        for path in sorted(p for p in base.rglob("*") if p.is_file()):
            final_manifest["files"].append({"path": str(path.relative_to(REPO_ROOT)).replace("\\", "/"), "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "bytes": path.stat().st_size})
final_manifest_path = REPO_ROOT / "phase5" / "validation" / "m1_shared_engine_final_hash_manifest.json"
final_manifest_path.write_text(json.dumps(final_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
archive_base = Path("/kaggle/working/phase5_m1_shared_engine_kaggle_proof")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=REPO_ROOT, base_dir="phase5"))
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
archive_hash_path = archive_path.with_suffix(archive_path.suffix + ".sha256")
archive_hash_path.write_text(f"{archive_sha256}  {archive_path.name}\n", encoding="ascii")
if not archive_path.is_file() or archive_path.stat().st_size == 0: raise RuntimeError("Proof ZIP creation failed")
NO_MORE_EXECUTION_AFTER_SYNC_OR_REVERIFY = True
assert NO_MORE_EXECUTION_AFTER_SYNC_OR_REVERIFY


In [ ]:
TERMINATION_GUARD = "no repository execution commands after sync/reverify in this seal epoch"
assert TERMINATION_GUARD


In [ ]:
if not archive_path.is_file(): raise RuntimeError("Proof ZIP is missing at termination")
if hashlib.sha256(archive_path.read_bytes()).hexdigest() != archive_sha256: raise RuntimeError("Proof ZIP hash changed after synchronization")
if archive_hash_path.read_text(encoding="ascii").split()[0] != archive_sha256: raise RuntimeError("Proof ZIP sidecar hash mismatch")
print(f"Proof complete: {archive_path.name} ({archive_sha256})")
